---
## 1. Imports & Path Configuration

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# ── Resolve project root regardless of where the kernel is launched ──
PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == 'notebooks'
    else Path.cwd().resolve()
)

# ── Input / output paths ──
CLEANED_PATH       = PROJECT_ROOT / 'data' / 'processed' / 'bigbasket_cleaned.csv'
TABLEAU_READY_PATH = PROJECT_ROOT / 'data' / 'processed' / 'bigbasket_tableau_ready.csv'
KPI_SUMMARY_PATH   = PROJECT_ROOT / 'data' / 'processed' / 'kpi_summary.csv'



---
## 2. Load Cleaned Dataset

In [5]:
df = pd.read_csv(CLEANED_PATH)

print(f'Dataset loaded successfully!')
print(f'Shape : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\nColumns ({df.shape[1]}):')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col} ({df[col].dtype})')

Dataset loaded successfully!
Shape : 27,200 rows × 15 columns

Columns (15):
   1. product (str)
   2. category (str)
   3. sub_category (str)
   4. brand (str)
   5. sale_price (float64)
   6. market_price (float64)
   7. type (str)
   8. rating (float64)
   9. description (str)
  10. is_price_anomaly (bool)
  11. discount_amount (float64)
  12. discount_pct (float64)
  13. price_segment (str)
  14. is_discounted (bool)
  15. rating_segment (str)


In [ ]:
# Quick sanity check — confirm no unexpected nulls in key columns
key_cols = ['product', 'category', 'sub_category', 'brand',
            'sale_price', 'market_price', 'rating',
            'discount_amount', 'discount_pct',
            'price_segment', 'is_discounted', 'rating_segment']

missing = df[key_cols].isnull().sum()
if missing.sum() == 0:
    print('✅  No missing values in key columns.')
else:
    print('⚠️  Missing values detected:')
    print(missing[missing > 0])

df[key_cols].head(5)

---
## 3. KPI Framework

The seven KPIs below are documented in `docs/data_dictionary.md` under
*KPI Column Reference* and appear on the Tableau dashboard.

| # | KPI | Description |
|---|-----|-------------|
| 1 | Average Discount (%) | Mean discount rate across all products |
| 2 | Average Sale Price (₹) | Mean selling price across the catalog |
| 3 | Average Rating | Mean consumer rating (out of 5) |
| 4 | % Products Discounted | Share of products priced below MRP |
| 5 | Top Category by Product Count | Category with the most listed SKUs |
| 6 | Top Brand by Avg Rating | Highest-rated brand (min 20 products) |
| 7 | Price Segment Distribution | Budget / Mid-range / Premium split |

In [6]:
# ── KPI 1: Average Discount Percentage ──
kpi_avg_discount_pct = df['discount_pct'].mean().round(2)

# ── KPI 2: Average Sale Price ──
kpi_avg_sale_price = df['sale_price'].mean().round(2)

# ── KPI 3: Average Rating ──
kpi_avg_rating = df['rating'].mean().round(2)

# ── KPI 4: % Products Discounted ──
kpi_pct_discounted = (df['is_discounted'].sum() / len(df) * 100).round(2)

# ── KPI 5: Top Category by Product Count ──
kpi_top_category = df['category'].value_counts().idxmax()
kpi_top_category_count = df['category'].value_counts().max()

# ── KPI 6: Top Brand by Avg Rating (min 20 products) ──
brand_rating = (
    df.groupby('brand')['rating']
    .agg(['mean', 'count'])
    .query('count >= 20')
    .sort_values('mean', ascending=False)
)
kpi_top_brand = brand_rating.index[0] if not brand_rating.empty else 'N/A'
kpi_top_brand_rating = brand_rating['mean'].iloc[0].round(2) if not brand_rating.empty else None

# ── KPI 7: Price Segment Distribution ──
kpi_price_segments = df['price_segment'].value_counts(normalize=True).mul(100).round(2)

print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print('         BIGBASKET KPI DASHBOARD SUMMARY     ')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'  KPI 1  Average Discount           : {kpi_avg_discount_pct} %')
print(f'  KPI 2  Average Sale Price          : ₹{kpi_avg_sale_price}')
print(f'  KPI 3  Average Rating              : {kpi_avg_rating} / 5')
print(f'  KPI 4  % Products Discounted       : {kpi_pct_discounted} %')
print(f'  KPI 5  Top Category                : {kpi_top_category} ({kpi_top_category_count:,} SKUs)')
print(f'  KPI 6  Top Brand (≥20 SKUs)        : {kpi_top_brand} ({kpi_top_brand_rating} avg rating)')
print('  KPI 7  Price Segment Distribution  :')
for seg, pct in kpi_price_segments.items():
    print(f'            {seg:<12} {pct} %')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
         BIGBASKET KPI DASHBOARD SUMMARY     
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  KPI 1  Average Discount           : 11.85 %
  KPI 2  Average Sale Price          : ₹320.95
  KPI 3  Average Rating              : 3.99 / 5
  KPI 4  % Products Discounted       : 55.32 %
  KPI 5  Top Category                : Beauty & Hygiene (7,682 SKUs)
  KPI 6  Top Brand (≥20 SKUs)        : Vahdam (4.47 avg rating)
  KPI 7  Price Segment Distribution  :
            Mid-range    56.8 %
            Budget       28.37 %
            Premium      14.83 %
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [7]:
# Save KPI summary to a machine-readable CSV
kpi_rows = [
    {'kpi': 'avg_discount_pct',   'value': kpi_avg_discount_pct,   'unit': '%'},
    {'kpi': 'avg_sale_price',     'value': kpi_avg_sale_price,     'unit': '₹'},
    {'kpi': 'avg_rating',         'value': kpi_avg_rating,         'unit': '/5'},
    {'kpi': 'pct_discounted',     'value': kpi_pct_discounted,     'unit': '%'},
    {'kpi': 'top_category',       'value': kpi_top_category,       'unit': 'name'},
    {'kpi': 'top_brand',          'value': kpi_top_brand,          'unit': 'name'},
    {'kpi': 'top_brand_avg_rating','value': kpi_top_brand_rating,  'unit': '/5'},
]

for seg, pct in kpi_price_segments.items():
    kpi_rows.append({'kpi': f'price_seg_{seg.lower().replace("-", "_")}', 'value': pct, 'unit': '%'})

kpi_df = pd.DataFrame(kpi_rows)
KPI_SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
kpi_df.to_csv(KPI_SUMMARY_PATH, index=False)
print(f'KPI summary saved → {KPI_SUMMARY_PATH}')
kpi_df

KPI summary saved → /Users/mac/Downloads/BigBasket_Market_Analysis/data/processed/kpi_summary.csv


,kpi,value,unit
0,avg_discount_pct,11.85,%
1,avg_sale_price,320.95,₹
2,avg_rating,3.99,/5
3,pct_discounted,55.32,%
4,top_category,Beauty & Hygiene,name
5,top_brand,Vahdam,name
6,top_brand_avg_rating,4.47,/5
7,price_seg_mid_range,56.80,%
8,price_seg_budget,28.37,%
9,price_seg_premium,14.83,%


---
## 4. Tableau Aggregation Tables

These aggregated views are what Tableau needs for the dashboard filters and charts.
They are embedded in the `bigbasket_tableau_ready.csv` export below,
but also shown here for review.

In [8]:
# ── Category-level aggregation ──
category_agg = (
    df.groupby('category')
    .agg(
        product_count   = ('product',        'count'),
        avg_sale_price  = ('sale_price',      'mean'),
        avg_market_price= ('market_price',    'mean'),
        avg_discount_pct= ('discount_pct',    'mean'),
        avg_rating      = ('rating',          'mean'),
        pct_discounted  = ('is_discounted',   'mean'),
    )
    .round(2)
    .reset_index()
    .sort_values('product_count', ascending=False)
)
category_agg['pct_discounted'] = (category_agg['pct_discounted'] * 100).round(2)

print(f'Category-level table: {category_agg.shape}')
category_agg.head(10)

Category-level table: (11, 7)


,category,product_count,avg_sale_price,avg_market_price,avg_discount_pct,avg_rating,pct_discounted
2,Beauty & Hygiene,7682,416.98,491.86,12.42,3.98,59.00
8,Gourmet & World Food,4677,320.11,358.72,7.86,4.04,40.00
9,"Kitchen, Garden & Pets",3457,508.79,663.52,22.56,3.78,78.00
10,Snacks & Branded Foods,2810,129.67,140.86,6.64,4.00,38.00
6,"Foodgrains, Oil & Masala",2673,193.28,230.29,11.85,4.07,60.00
4,Cleaning & Household,2650,226.48,262.44,10.78,3.99,56.00
3,Beverages,884,239.97,272.49,9.58,4.12,44.00
1,"Bakery, Cakes & Dairy",850,142.84,157.89,7.64,3.93,49.00
0,Baby Care,610,534.95,596.75,5.85,4.06,36.00
7,Fruits & Vegetables,557,50.89,64.43,21.25,4.10,97.00


In [9]:
# ── Brand-level aggregation (top 30 by product count) ──
brand_agg = (
    df.groupby('brand')
    .agg(
        product_count   = ('product',      'count'),
        avg_sale_price  = ('sale_price',   'mean'),
        avg_discount_pct= ('discount_pct', 'mean'),
        avg_rating      = ('rating',       'mean'),
    )
    .round(2)
    .reset_index()
    .sort_values('product_count', ascending=False)
    .head(30)
)

print(f'Brand-level table (top 30): {brand_agg.shape}')
brand_agg.head(10)

Brand-level table (top 30): (30, 5)


,brand,product_count,avg_sale_price,avg_discount_pct,avg_rating
747,Fresho,638,85.02,21.43,4.10
208,Bb Royal,539,211.61,25.90,4.07
205,Bb Home,428,233.74,33.11,4.01
586,Dp,243,294.38,28.57,4.06
749,Fresho Signature,170,296.24,13.30,4.07
204,Bb Combo,167,479.46,18.80,4.09
96,Amul,153,155.23,5.18,4.04
976,Inatur,142,404.44,28.98,3.91
928,Himalaya,141,225.70,12.94,4.14
505,Dabur,137,179.51,10.86,4.11


In [10]:
# ── Price segment × Category cross-tab ──
segment_cat = (
    df.groupby(['category', 'price_segment'], observed=True)
    .agg(
        product_count   = ('product',  'count'),
        avg_rating      = ('rating',   'mean'),
        avg_discount_pct= ('discount_pct', 'mean'),
    )
    .round(2)
    .reset_index()
)

print(f'Segment × Category cross-tab: {segment_cat.shape}')
segment_cat.head(10)

Segment × Category cross-tab: (33, 5)


,category,price_segment,product_count,avg_rating,avg_discount_pct
0,Baby Care,Budget,60,4.16,3.77
1,Baby Care,Mid-range,333,4.02,3.70
2,Baby Care,Premium,217,4.08,9.72
3,"Bakery, Cakes & Dairy",Budget,403,4.01,7.15
4,"Bakery, Cakes & Dairy",Mid-range,431,3.86,8.02
5,"Bakery, Cakes & Dairy",Premium,16,3.86,9.65
6,Beauty & Hygiene,Budget,1318,4.09,7.80
7,Beauty & Hygiene,Mid-range,4753,3.98,13.67
8,Beauty & Hygiene,Premium,1611,3.91,12.50
9,Beverages,Budget,242,4.06,8.70


In [11]:
# ── Rating segment distribution per category ──
rating_seg_cat = (
    df.groupby(['category', 'rating_segment'], observed=True)
    .agg(product_count=('product', 'count'))
    .reset_index()
)

print(f'Rating segment × Category: {rating_seg_cat.shape}')
rating_seg_cat.head(10)

Rating segment × Category: (29, 3)


,category,rating_segment,product_count
0,Baby Care,High,417
1,Baby Care,Low,56
2,Baby Care,Medium,137
3,"Bakery, Cakes & Dairy",High,310
4,"Bakery, Cakes & Dairy",Low,42
5,"Bakery, Cakes & Dairy",Medium,498
6,Beauty & Hygiene,High,5336
7,Beauty & Hygiene,Low,729
8,Beauty & Hygiene,Medium,1617
9,Beverages,High,692


---
## 5. Build & Export Tableau-Ready Dataset

The Tableau-ready CSV is the **full cleaned dataset** with all engineered columns,
plus a `category_rank` column that lets Tableau filter by popularity.
`price_segment` and `rating_segment` are cast to string so Tableau treats them
as discrete dimensions rather than measures.

In [12]:
tableau_df = df.copy()

# Tableau-friendly type conversions
tableau_df['price_segment']  = tableau_df['price_segment'].astype(str)
tableau_df['rating_segment'] = tableau_df['rating_segment'].astype(str)
tableau_df['is_discounted']  = tableau_df['is_discounted'].map({True: 'Yes', False: 'No'})
tableau_df['is_price_anomaly'] = tableau_df['is_price_anomaly'].map({True: 'Yes', False: 'No'})

# Add a category popularity rank (1 = most products)
cat_rank = (
    df['category'].value_counts()
    .rank(ascending=False, method='min')
    .astype(int)
    .rename('category_rank')
)
tableau_df = tableau_df.merge(cat_rank, left_on='category', right_index=True, how='left')

print(f'Tableau-ready dataset shape: {tableau_df.shape}')
print(f'Columns: {tableau_df.columns.tolist()}')
tableau_df.head(3)

Tableau-ready dataset shape: (27200, 16)
Columns: ['product', 'category', 'sub_category', 'brand', 'sale_price', 'market_price', 'type', 'rating', 'description', 'is_price_anomaly', 'discount_amount', 'discount_pct', 'price_segment', 'is_discounted', 'rating_segment', 'category_rank']


,product,category,sub_category,brand,sale_price,market_price,type,rating,description,is_price_anomaly,discount_amount,discount_pct,price_segment,is_discounted,rating_segment,category_rank
0,Garlic Oil - Vegetarian Capsule 500 mg,Beauty & Hygiene,Hair Care,Sri Sri Ayurveda,220.00,220.00,Hair Oil & Serum,4.10,This Product contains Garlic Oil that is known...,No,0.00,0.00,Mid-range,No,High,1
1,Water Bottle - Orange,"Kitchen, Garden & Pets",Storage & Accessories,Mastercook,180.00,180.00,Water & Fridge Bottles,2.30,"Each product is microwave safe (without lid), ...",No,0.00,0.00,Mid-range,No,Low,3
2,"Brass Angle Deep - Plain, No.2",Cleaning & Household,Pooja Needs,Trm,119.00,250.00,Lamp & Lamp Oil,3.40,"A perfect gift for all occasions, be it your m...",No,131.00,52.40,Mid-range,Yes,Medium,6


In [13]:
# Export
TABLEAU_READY_PATH.parent.mkdir(parents=True, exist_ok=True)
tableau_df.to_csv(TABLEAU_READY_PATH, index=False)
print(f'✅  Tableau-ready dataset saved → {TABLEAU_READY_PATH}')
print(f'   Rows    : {len(tableau_df):,}')
print(f'   Columns : {len(tableau_df.columns)}')

✅  Tableau-ready dataset saved → /Users/mac/Downloads/BigBasket_Market_Analysis/data/processed/bigbasket_tableau_ready.csv
   Rows    : 27,200
   Columns : 16


---
## 6. Final Submission-Ready Summary

Run this cell last to confirm all pipeline outputs are in place before committing.

In [14]:
def check_file(path: Path) -> str:
    if path.exists():
        size_kb = path.stat().st_size / 1024
        return f'✅  {path.name:<45}  ({size_kb:,.0f} KB)'
    return f'❌  {path.name:<45}  MISSING'

files_to_check = [
    PROJECT_ROOT / 'data' / 'raw'       / 'BigBasket_Products.csv',
    PROJECT_ROOT / 'data' / 'processed' / 'bigbasket_cleaned.csv',
    PROJECT_ROOT / 'data' / 'processed' / 'bigbasket_tableau_ready.csv',
    PROJECT_ROOT / 'data' / 'processed' / 'kpi_summary.csv',
    PROJECT_ROOT / 'docs'               / 'data_dictionary.md',
    PROJECT_ROOT / 'scripts'            / 'etl_pipeline.py',
    PROJECT_ROOT / 'notebooks'          / '01_extraction.ipynb',
    PROJECT_ROOT / 'notebooks'          / '02_cleaning.ipynb',
    PROJECT_ROOT / 'notebooks'          / '03_eda.ipynb',
    PROJECT_ROOT / 'notebooks'          / 'stat_analysis.ipynb',
    PROJECT_ROOT / 'notebooks'          / 'final_load_repo.ipynb',
    PROJECT_ROOT / 'README.md',
]

print('\n========== SUBMISSION FILE CHECK ==========')
for f in files_to_check:
    print(check_file(f))
print('============================================')
print()
print('Pipeline complete. All outputs ready for Tableau and final report.')


========== SUBMISSION FILE CHECK ==========
✅  BigBasket_Products.csv                         (16,347 KB)
✅  bigbasket_cleaned.csv                          (17,036 KB)
✅  bigbasket_tableau_ready.csv                    (16,960 KB)
✅  kpi_summary.csv                                (0 KB)
✅  data_dictionary.md                             (5 KB)
✅  etl_pipeline.py                                (7 KB)
✅  01_extraction.ipynb                            (27 KB)
✅  02_cleaning.ipynb                              (34 KB)
✅  03_eda.ipynb                                   (1,339 KB)
✅  stat_analysis.ipynb                            (847 KB)
✅  final_load_repo.ipynb                          (47 KB)
✅  README.md                                      (9 KB)

Pipeline complete. All outputs ready for Tableau and final report.
